# Aula 22 — Processamento Batch vs. Streaming

**Disciplina:** Big Data e Inteligência Artificial  
**Data:** 17/11/2026  
**Fonte:** `samples.wanderbricks.clickstream` — eventos de navegação disponibilizados pelo Databricks  
**Ambiente:** Databricks Free Edition (compute serverless)

## Objetivo

Comparar processamento **batch** e **streaming** utilizando os **mesmos eventos**, em cinco ciclos progressivos: leitura batch, chegada de dados, streaming de tabela Delta, retomada com checkpoint e ingestão de arquivos com Auto Loader.

**Importante:** os eventos são registros de uma tabela histórica do catálogo `samples`. Nós apenas **reproduzimos sua chegada** em uma tabela Delta própria; não existe uma fonte online de eventos nesta aula. Na aula seguinte, o caso será MQTT real.

**Roteiro:** execute as células em ordem. As atividades `TODO` são resolvidas pelos alunos na versão de estudante.

**Versão do aluno:** complete as atividades `TODO`.

## Preparação — catálogo `samples`, tabelas de trabalho e checkpoint

O catálogo `samples` é **somente leitura**: não o modificaremos. Copiaremos **30 eventos existentes** para uma amostra de trabalho, com um número sequencial `id_evento` acrescentado apenas para organizar os três lotes didáticos.

As tabelas de trabalho serão criadas no **seu catálogo e esquema**, e o checkpoints e arquivos JSON do Ciclo 5 ficarão em um **Volume do Unity Catalog**.

**Antes de executar:** confira o `catalogo` e o `esquema` na célula. Se o catálogo atual não permitir criação de tabelas/Volumes, substitua pelos nomes em que sua conta tem permissão.

**Reinício:** a célula de preparação recria apenas os objetos identificados como `aula22_*`. Para refazer a experiência, execute esta preparação e todos os ciclos na ordem. **Não reexecute apenas uma célula que adiciona um lote**, pois isso poderá inserir eventos duplicados na fonte.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Altere somente se seu workspace utilizar outro catálogo ou esquema.
catalogo = spark.sql("SELECT current_catalog()").first()[0]
esquema = "default"

origem_samples = "samples.wanderbricks.clickstream"
amostra = f"`{catalogo}`.`{esquema}`.`aula22_amostra`"
fonte = f"`{catalogo}`.`{esquema}`.`aula22_fonte`"
destino = f"`{catalogo}`.`{esquema}`.`aula22_destino`"
volume = "aula22_streaming"

# Checkpoint exclusivo da atividade.
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalogo}`.`{esquema}`.`{volume}`")
checkpoint = f"/Volumes/{catalogo}/{esquema}/{volume}/checkpoint_clickstream"

# Reinicia somente os objetos da aula.
dbutils.fs.rm(checkpoint, True)
spark.sql(f"DROP TABLE IF EXISTS {destino}")
spark.sql(f"DROP TABLE IF EXISTS {fonte}")
spark.sql(f"DROP TABLE IF EXISTS {amostra}")

# Seleciona registros da tabela pública. O id_evento é criado para dividir os lotes.
base = (spark.table(origem_samples)
    .select(F.col("event").alias("evento"),
            F.col("metadata.device").alias("dispositivo"))
    .where(F.col("evento").isNotNull())
    .limit(120)
    .withColumn("id_evento", F.row_number().over(Window.orderBy(F.rand(seed=22))))
    .filter(F.col("id_evento") <= 30)
    .select("id_evento", "evento", "dispositivo"))

base.write.format("delta").mode("overwrite").saveAsTable(amostra)

# Tabela Delta vazia: receberá os eventos à medida que liberarmos cada lote.
spark.table(amostra).limit(0).write.format("delta").mode("overwrite").saveAsTable(fonte)

print("Fonte pública:", origem_samples)
print("Tabela de entrada:", fonte)
print("Tabela de saída:", destino)
print("Eventos selecionados:", spark.table(amostra).count())
display(spark.table(amostra).orderBy("id_evento"))

## Ciclo 1 — Leitura batch

**Conceito:** processar os registros disponíveis em uma execução delimitada.

Vamos liberar os **12 primeiros eventos** na tabela de entrada e fazer uma leitura batch. O resultado calculado nesta etapa não muda sozinho quando novos registros chegarem.

In [ ]:
# Primeira chegada: 12 eventos do dataset público.
lote1 = spark.table(amostra).filter(F.col("id_evento").between(1, 12))
lote1.write.format("delta").mode("append").saveAsTable(fonte)

# Processamento batch.
df_batch_inicial = spark.read.table(fonte)
total_inicial = df_batch_inicial.count()
print("Quantidade de eventos na primeira análise batch:", total_inicial)
display(df_batch_inicial.orderBy("id_evento"))

**Atividade 1 — Eventos por tipo**

Usando `df_batch_inicial`, calcule a quantidade de eventos para cada valor de `evento`, em ordem decrescente de frequência. Qual evento aparece mais vezes neste primeiro conjunto?

In [ ]:
# Atividade 1
# TODO: agrupe df_batch_inicial pela coluna "evento".
# TODO: conte os registros por grupo e ordene da maior para a menor contagem.
# TODO: exiba o resultado.

## Ciclo 2 — Novos eventos e nova leitura batch

**Conceito:** uma operação batch também pode ser executada novamente. A diferença é que **o resultado já calculado não se atualiza por si só**.

Vamos acrescentar mais **8 eventos** à mesma tabela de entrada.

In [ ]:
# Segunda chegada: mais 8 eventos (IDs 13 a 20).
lote2 = spark.table(amostra).filter(F.col("id_evento").between(13, 20))
lote2.write.format("delta").mode("append").saveAsTable(fonte)

print("Resultado da primeira análise batch (já calculado):", total_inicial)

df_batch_atualizado = spark.read.table(fonte)
print("Nova leitura batch:", df_batch_atualizado.count())
display(df_batch_atualizado.orderBy("id_evento"))

**Atividade 2 — Atualização dos indicadores**

Na nova leitura, conte os eventos por `dispositivo`. Compare o total atual com `total_inicial` e explique por que os dois valores diferem.

In [ ]:
# Atividade 2
# TODO: agrupe df_batch_atualizado por "dispositivo".
# TODO: conte os eventos e exiba o resultado.
# TODO: escreva sua explicação em uma célula Markdown.

## Ciclo 3 — Structured Streaming em tabela Delta

**Conceito:** `readStream` acompanha uma origem que admite novos registros. O checkpoint registra o progresso da consulta. A primeira execução processa os **20 eventos já disponíveis**.

No compute serverless, utilizaremos `.trigger(availableNow=True)`: a consulta processa os dados disponíveis **e encerra**. Isso é streaming **incremental sob demanda**, não uma execução permanentemente ativa.

In [ ]:
# Lê incrementalmente a tabela Delta de entrada.
df_stream = spark.readStream.table(fonte)

consulta = (df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(destino))

consulta.awaitTermination()

print("Eventos persistidos pelo streaming:", spark.table(destino).count())
display(spark.table(destino).orderBy("id_evento"))

**Atividade 3 — Consultando a saída do streaming**

Consulte a tabela `destino` com `spark.table()` e conte os eventos por `evento`. Explique: a tabela `samples` está sendo lida continuamente, ou a consulta acompanha a **tabela Delta de entrada** que estamos alimentando?

In [ ]:
# Atividade 3
# TODO: leia spark.table(destino).
# TODO: calcule a quantidade de registros por "evento" e mostre o resultado.
# TODO: responda a questão em uma célula Markdown.

## Ciclo 4 — Retomada com o mesmo checkpoint

**Conceito:** a próxima execução do streaming deve consumir somente os eventos acrescentados desde o último processamento.

Vamos liberar os **10 eventos restantes** e executar a mesma consulta, com o **mesmo caminho de checkpoint** e a mesma tabela de destino.

In [ ]:
# Terceira chegada: mais 10 eventos (IDs 21 a 30).
lote3 = spark.table(amostra).filter(F.col("id_evento").between(21, 30))
lote3.write.format("delta").mode("append").saveAsTable(fonte)

# A consulta retoma do checkpoint anterior.
consulta = (spark.readStream.table(fonte)
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(destino))

consulta.awaitTermination()

print("Total da fonte:", spark.table(fonte).count())
print("Total do destino:", spark.table(destino).count())

**Atividade 4 — Verificando a ingestão incremental**

Compare as quantidades de registros na fonte e no destino. Calcule também quantos `id_evento` distintos existem no destino. Por que a segunda execução não deve produzir **50 registros** (20 antigos + 30 totais)?

In [ ]:
# Atividade 4
# TODO: conte os registros de spark.table(fonte) e spark.table(destino).
# TODO: conte os id_evento distintos no destino.
# TODO: explique o papel do checkpoint em uma célula Markdown.

## Ciclo 5 — Ingestão de arquivos com Auto Loader

Até agora, o streaming consumiu uma **tabela Delta**. Agora utilizaremos o **Auto Loader**, que identifica e processa novos **arquivos** em uma pasta.

A demonstração grava **10 eventos em JSON** e utiliza `cloudFiles` para carregá-los em outra tabela Delta. O checkpoint permite continuar a ingestão quando chegarem novos arquivos.

In [ ]:
# Preparação da pasta de arquivos e da tabela de saída
pasta_json = f"/Volumes/{catalogo}/{esquema}/{volume}/ciclo5_json"
checkpoint_autoloader = f"/Volumes/{catalogo}/{esquema}/{volume}/checkpoint_autoloader"
tabela_autoloader = f"`{catalogo}`.`{esquema}`.`aula22_autoloader`"

# Reinicia somente os objetos deste ciclo.
dbutils.fs.rm(checkpoint_autoloader, True)
dbutils.fs.rm(pasta_json, True)
spark.sql(f"DROP TABLE IF EXISTS {tabela_autoloader}")

# Disponibiliza os primeiros 10 eventos como arquivos JSON.
(spark.table(amostra)
    .filter(F.col("id_evento").between(1, 10))
    .write.mode("overwrite").json(pasta_json))

# Auto Loader: descobre arquivos JSON e grava os registros em Delta.
consulta_arquivos = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(spark.table(amostra).schema)
    .load(pasta_json)
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_autoloader)
    .trigger(availableNow=True)
    .toTable(tabela_autoloader))

consulta_arquivos.awaitTermination()
print("Eventos ingeridos por Auto Loader:", spark.table(tabela_autoloader).count())
display(spark.table(tabela_autoloader).orderBy("id_evento"))

**Atividade 5 — Novos arquivos JSON**

Grave os eventos de `id_evento` **11 a 15** na mesma pasta `pasta_json`, usando a escrita em modo `append`. Em seguida, execute novamente o Auto Loader com **o mesmo checkpoint** e confirme que a tabela `tabela_autoloader` contém **15 eventos**, sem duplicar os 10 anteriores.

**Atenção:** execute a atividade uma vez após a demonstração. Para reiniciar, execute novamente o início do Ciclo 5.

In [ ]:
# Atividade 5
# TODO: selecione os eventos de 11 a 15 na tabela amostra e grave em pasta_json com append.
# TODO: execute novamente readStream.format("cloudFiles"), com a mesma pasta e schema.
# TODO: utilize checkpoint_autoloader e tabela_autoloader, com availableNow=True.
# TODO: confirme que há 15 registros na tabela de saída.

---
**Referências oficiais:**
- [Wanderbricks — dataset do catálogo samples](https://docs.databricks.com/aws/en/discover/wanderbricks-dataset)
- [Structured Streaming lendo e escrevendo em Delta Lake](https://docs.databricks.com/aws/en/structured-streaming/delta-lake)
- [Streaming em compute serverless: availableNow](https://docs.databricks.com/aws/en/compute/serverless/streaming)
- [Auto Loader — ingestão incremental de arquivos](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader)